# Case study 1:  Multimodal accessibility to health stations in Helsinki, Finland

This notebook demonstrates how to quickly compute the travel times on public transport, by bicycle, or on foot from each of the 1800 population register grid cells to the closest one of the 20 health stations distributed across the city. 

The script downloads the population grid and the health station locations from open data WFS endpoints, and fetches the public transport schedule from the local transport authority’s open data portal.

In [ ]:
import urllib.parse

import geopandas
import shapely
import r5py

In [ ]:
BOUNDING_BOX = shapely.box(25490584, 6669667, 25508835, 6682088)

### 1. Download a census grid, used as origin locations

The population grid used here is available as an open government data (OGD) set under a *Creative Commons Attribution 4.0 license*, from https://hri.fi/data/en_GB/dataset/vaestotietoruudukko. It features population counts by age group for 2022.

We modify the dataset in two minor steps before further processing:

1. To speed up `geopandas.GeoDataFrame.explore()`, we reproject the data into a WebMercator/EPSG:3857 reference system
2. As r5py requires an `id` column and the data set does not have such a column, we create it

Beyond that, we clip the data set to the bounding box defined above.

In [ ]:
population_grid = geopandas.read_file(
    "https://kartta.hsy.fi/geoserver/wfs?"
    + urllib.parse.urlencode(
        {
            "service": "WFS",
            "version": "2.0.0",
            "request": "GetFeature",
            "typeName": "asuminen_ja_maankaytto:Vaestotietoruudukko_2022",
            "srsName": "EPSG:3879",
            "bbox": f"{",".join([f"{x:0.0f}" for x in BOUNDING_BOX.bounds])},EPSG:3879",
        }
    )
).set_crs("EPSG:3879").to_crs("EPSG:3857")
population_grid["id"] = population_grid["index"] 

population_grid.explore()

### 2. Download the locations of Helsinki’s health stations

Also the locations of health stations, as well as all other public services provided by the city of Helsinki, are available as an OGD data set, under the same *Creative Commons Attribution 4.0 license*, from https://hri.fi/data/en_GB/dataset/paakaupunkiseudun-palvelukartan-rest-rajapinta. The remote dataset contains all services provided in the entire metropolitan region, which is why we use an [FES](https://www.ogc.org/standards/filter/) filter expression to fetch health stations, only. As with the census grid, we then proceed to clip the data set to the bounding box, reproject it to Web Mercator, and assign an `id` column.

In [ ]:
health_stations = geopandas.read_file(
    "https://kartta.hel.fi/ws/geoserver/avoindata/wfs?"
    + urllib.parse.urlencode(
        {
            "service": "WFS",
            "version": "2.0.0",
            "request": "GetFeature",
            "typeName": "avoindata:Toimipisterekisteri_palvelut",
            "Filter": "<Filter><PropertyIsEqualTo><ValueReference>avoindata:service_en</ValueReference><Literal>health stations</Literal></PropertyIsEqualTo></Filter>",
            "srsName": "EPSG:3879",
        }
    )
).set_crs("EPSG:3879").clip(BOUNDING_BOX).to_crs("EPSG:3857")
health_stations["id"] = health_stations["unit_id"]

health_stations.explore()

### 3. Initialise a transport network model with open data

Next, we create a new model of the transport network of the city. For this, we need two data sources, both available openly:

- an OpenStreetMap (OSM) extract, with details on the street, bicycle, and footpath network, downloaded from BBBIKE (https://download.bbbike.org/osm/), under the Open Database License (https://opendatacommons.org/licenses/odbl/summary/)
- a public transport schedule in GTFS format, provided by Helsingin Seudun Liikkenne (the public transport authority) under a Creative Commons Attribution 4.0 license (https://www.hsl.fi/en/hsl/open-data#public-transport-network-and-timetables-gtfs)

In [ ]:
transport_network = r5py.TransportNetwork(
    "https://download.bbbike.org/osm/bbbike/Helsinki/Helsinki.osm.pbf",  # OpenStreetMap extract
    ["https://infopalvelut.storage.hsldev.com/gtfs/hsl.zip"],  # public transport schedule
)


### 4. Compute travel times 

For better code readability, we create two slim data frames that contain only the data needed for travel time computation, for origins and destinations, respectively. For the census grid, we decide to use the centroid point of each cell.

Then, we initialise a travel time matrix, passing four parameters:

1. the transport network model created above,
2. origin locations,
3. destination locations, and
4. the modes of transport r5py will query, reporting the shortest travel time, only

In [ ]:
origins = population_grid[["id", "geometry"]]
origins["geometry"] = origins["geometry"].centroid

destinations = health_stations[["id", "geometry"]]

In [ ]:
travel_times_to_health_stations = r5py.TravelTimeMatrix(
    transport_network, 
    origins, 
    destinations, 
    transport_modes=[
        r5py.TransportMode.TRANSIT, 
        r5py.TransportMode.WALK, 
        r5py.TransportMode.BICYCLE,
    ]
)

In [ ]:
travel_times_to_health_stations

### 5. Interpret and visualise results

As you can see from the table listing above, the result data set is a list of travel times from each origin to each destination. We can now reduce this data set to retain only the travel time to the health station closest to each census grid centroid, before joining the travel times back to the original population grid for visualisation and for further analysis (e.g., weighting the results by the share of older residents).

In [ ]:
travel_times_to_health_stations = travel_times_to_health_stations.loc[
    travel_times_to_health_stations
        .dropna()
        .groupby("from_id")["travel_time"].idxmin()
]

In [ ]:
population_grid.set_index("id").join(travel_times_to_health_stations.set_index("from_id")).reset_index().explore("travel_time", cmap="Reds")